### Set path directory

In [1]:
from pathlib import Path
local_dir = Path.cwd() / "data"
local_dir.mkdir(parents=True, exist_ok=True)

### Import SoccerNet

In [2]:
import SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader

mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=str(local_dir))
mySoccerNetDownloader.password = "s0cc3rn3t"

### Download data

In [3]:
mySoccerNetDownloader.downloadDataTask(task="tracking-2023", split=["train", "test", "challenge"])
mySoccerNetDownloader.downloadDataTask(task="jersey-2023", split=["train","test","challenge"])

### UnZIP files

### Visualise football clips
#### Tracking-2023

In [7]:
import os, cv2, time
from pathlib import Path
from collections import defaultdict

folder_name = "SNMOT-104"

def find_folder_downwards(target_folder: str, start_dir: Path | None = None) -> Path | None:
    start_dir = start_dir or local_dir
    for path in start_dir.rglob(target_folder):
        if path.is_dir():
            return path
    return None

# --- Locate sequence ---
file_dir = find_folder_downwards(folder_name)
if file_dir is None:
    raise FileNotFoundError(f"Folder '{folder_name}' not found from {local_dir}")

seqdir = Path(file_dir)
imgdir = seqdir / "img1"
gt_dir = seqdir / "gt"
gt_txt = gt_dir / "gt.txt"

# --- Build per-frame GT (if available) ---
per_frame = defaultdict(list)
if gt_txt.is_file():
    with open(gt_txt) as f:
        for line in f:
            if not line.strip():
                continue
            fr, tid, x, y, w, h, conf, *rest = line.strip().split(",")
            cls_ = int(float(rest[0])) if rest else 1
            per_frame[int(float(fr))].append(
                (int(float(tid)), cls_,
                 int(float(x)), int(float(y)), int(float(w)), int(float(h)))
            )

# --- Playback settings ---
win = "SNMOT preview"
target_fps = 50  # max 50
frame_period = 1.0 / target_fps

# OpenCV image-sequence reader; expects 000001.jpg, 000002.jpg, ...
cap = cv2.VideoCapture(str(imgdir / "%06d.jpg"))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open image sequence at {imgdir}")

window_created = False
i = 1

try:
    while True:
        t0 = time.perf_counter()

        ok, img = cap.read()
        if not ok or img is None:
            break

        # Draw boxes if GT exists
        if per_frame:
            for tid, cls_, x, y, w, h in per_frame.get(i, []):
                color = (0, 0, 225) if tid == 19 else (225, 225, 0)  # ball red, players cyan - FIX for ball identification
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv2.putText(img, f"{tid}", (x, max(0, y - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

        # Create window when first frame is ready
        if not window_created:
            cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)
            window_created = True

        cv2.imshow(win, img)

        # Adaptive delay to hit target FPS
        elapsed = time.perf_counter() - t0
        remaining = frame_period - elapsed
        delay_ms = 1 if remaining <= 0 else int(remaining * 1000)

        key = cv2.waitKey(delay_ms) & 0xFF
        if key in (27, ord('q')):  # ESC or q
            break

        i += 1
finally:
    cap.release()
    cv2.destroyWindow(win)
    cv2.waitKey(1)  # let macOS process the close


#### Jersey-2023

In [13]:
import tempfile
import re
import numpy as np

data_split = "train"    # "train", "test", "challenge"
folder_number = "97"   # 0 - 1425 (challenge), 1210 (test), train (1426)

seqdir = Path(local_dir)
imgdir = seqdir / "jersey-2023" / data_split / "images" / folder_number


# Grab all matching images and sort them by frame number
images = sorted(imgdir.glob(f"{folder_number}_*.jpg"),
                key=lambda p: int(p.stem.split("_")[1]))


# --- Playback settings ---
win = "Clip preview"
target_fps = 50  # max 50
frame_period = 1.0 / target_fps

cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)

try:
    for i, img_path in enumerate(images, start=1):
        t0 = time.perf_counter()
        img = cv2.imread(str(img_path))
        if img is None:
            continue


        cv2.imshow(win, img)

        elapsed = time.perf_counter() - t0
        remaining = frame_period - elapsed
        delay_ms = 1 if remaining <= 0 else int(remaining * 1000)

        key = cv2.waitKey(delay_ms) & 0xFF
        if key in (27, ord('q')):
            break
finally:
    cv2.destroyWindow(win)
    cv2.waitKey(1)